# Rectangular waveguide modes

*em · advanced* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/em/waveguide_modes)

Which field patterns can travel down a hollow metal pipe? Each mode solves the 2D Helmholtz equation $(\partial_i\partial_i + k_c^2)\psi = 0$ on the cross-section — Dirichlet walls for TM, Neumann walls for TE. The discrete Laplacian is assembled with torch.kron and torch.linalg.eigh returns all modes and cutoff frequencies at once.

**Vector form**

$$(\nabla_\perp^2 + k_c^2)\,\psi = 0,\qquad \psi = E_z\ (\mathrm{TM}:\ \psi|_{\partial} = 0),\quad \psi = H_z\ (\mathrm{TE}:\ \partial_n\psi|_{\partial} = 0)$$

**Index (tensor) form**

$$\partial_i\partial_i\,\psi + k_c^2\,\psi = 0 \;\;\longrightarrow\;\; -L_{nm}\,\psi_m = k_c^2\,\psi_n,\qquad L = D_x\otimes I_y + I_x\otimes D_y$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $\psi_{ab}$ | `pattern` | (Nx, Ny) | $E_z$ (TM) or $H_z$ (TE) at grid point $(x_a, y_b)$ |
| $\psi_m$ | `pattern.reshape(-1)` | (Nx·Ny,) | same values flattened: $m = a\cdot N_y + b$ |
| $D_x,\ D_y$ | `second_difference(n, h, boundary)` | (Nx, Nx), (Ny, Ny) | 1D $\partial^2$ with the wall condition built into the corner entries |
| $L_{nm}$ | `kron(Dx, Iy) + kron(Ix, Dy)` | (Nx·Ny, Nx·Ny) | the 2D Laplacian $\partial_i\partial_i$ ($i$ summed over $x, y$) |
| $k_c^2,\ \psi_m$ | `torch.linalg.eigh(-L)` | (Nx·Ny,), (Nx·Ny, Nx·Ny) | eigenvalues and orthonormal eigenvectors: cutoffs and mode shapes |
| $k_c^{mn}$ | `exact_cutoff(m, n, a, b)` | scalar | $\pi\sqrt{(m/a)^2 + (n/b)^2}$, to check the numbers |

Default size is the standard WR-90 X-band guide (2.286 cm × 1.016 cm), whose $\text{TE}_{10}$ mode cuts off at 6.56 GHz. A mode propagates only above its cutoff, with $\beta = \sqrt{(\omega/c)^2 - k_c^2}$. TE and TM modes with the same $(m, n \ge 1)$ share a cutoff. The constant TE solution ($k_c = 0$) carries no wave and is dropped. Second-order finite differences put $k_c$ within about a percent of the exact value at the default resolution.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/em/waveguide_modes/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli

from . import plot, solve


def run(params: dict, out_dir: Path) -> Path:
    a, b = params["a"], params["b"]  # cm   dense eigenproblems: CPU float64

    found = solve.solve(a, b, params["n_grid"], params["n_modes"])

    frequencies = [solve.cutoff_frequency_GHz(mode["k_c"]) for mode in found]
    for mode, f in zip(found, frequencies):
        error = abs(mode["k_c"] - mode["k_exact"]) / mode["k_exact"]
        print(f"{mode['kind']}{mode['mn']}  k_c = {mode['k_c']:.4f} /cm  (exact {mode['k_exact']:.4f}, "
              f"error {error:.1e})   f_c = {f:.2f} GHz")  # fmt: skip
    return plot.animate(found, frequencies, a, b, out_dir / "waveguide_modes.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/em/waveguide_modes/equation.py
r"""Physics → tensor translation.

Inside a hollow metal pipe of cross-section a × b, a wave travelling along z has the form
ψ(x, y) e^{i(βz − ωt)}. Maxwell's equations reduce to a 2D Helmholtz eigenproblem for ψ:

    (\partial_i \partial_i + k_c^2)\, \psi = 0,      i ∈ {x, y} summed,      β^2 = (ω/c)^2 - k_c^2

    TM modes: ψ = E_z, which must vanish on the metal walls       (Dirichlet:  ψ = 0)
    TE modes: ψ = H_z, whose normal derivative vanishes there      (Neumann:    ∂_n ψ = 0)

Below the cutoff frequency f_c = c k_c / 2π the mode cannot propagate (β is imaginary).

On an Nx × Ny grid the 1D second derivative is a tridiagonal matrix D. Flatten ψ_{ab} (a = x, b = y)
into ψ_m with m = a·Ny + b; then ∂_x² acts on the first index and ∂_y² on the second, a Kronecker sum:

    L = D_x ⊗ I_y + I_x ⊗ D_y,         L_{nm} ψ_m = -k_c^2 ψ_n       (m summed)

So the modes are the eigenvectors of the symmetric matrix −L and k_c² its eigenvalues: one call to
torch.linalg.eigh. The exact answer for a rectangle is k_c = π √((m/a)² + (n/b)²).
"""

import math

import torch


def second_difference(n: int, h: float, boundary: str) -> torch.Tensor:
    """D of shape (N, N): (ψ_{a−1} − 2ψ_a + ψ_{a+1}) / h² with the walls built in.

    dirichlet: grid nodes strictly inside the walls; the missing neighbours are ψ = 0, so they drop out.
    neumann:   grid cells centred half a cell from the walls; the missing neighbour mirrors ψ_a (zero slope),
               so the corner entries become −1 instead of −2.
    """
    D = torch.diag(-2.0 * torch.ones(n, dtype=torch.float64))
    D += torch.diag(torch.ones(n - 1, dtype=torch.float64), 1) + torch.diag(torch.ones(n - 1, dtype=torch.float64), -1)
    if boundary == "neumann":
        D[0, 0] = D[-1, -1] = -1.0
    return D / h**2


def laplacian(nx: int, ny: int, hx: float, hy: float, boundary: str) -> torch.Tensor:
    """L = D_x ⊗ I_y + I_x ⊗ D_y of shape (Nx·Ny, Nx·Ny)."""
    Ix = torch.eye(nx, dtype=torch.float64)
    Iy = torch.eye(ny, dtype=torch.float64)
    return torch.kron(second_difference(nx, hx, boundary), Iy) + torch.kron(Ix, second_difference(ny, hy, boundary))


def modes(L: torch.Tensor, count: int, skip_zero: bool) -> tuple[torch.Tensor, torch.Tensor]:
    r"""Solve −L_{nm} ψ_m = k_c² ψ_n.   Returns (k_c (K,), ψ (Nx·Ny, K)), lowest k_c first, orthonormal columns.

    skip_zero drops the constant Neumann solution (k_c = 0): a constant H_z carries no wave.
    """
    k2, psi = torch.linalg.eigh(-L)  # eigenvalues in ascending order
    start = 1 if skip_zero else 0
    return torch.sqrt(k2[start : start + count].clamp(min=0)), psi[:, start : start + count]


def exact_cutoff(m: int, n: int, a: float, b: float) -> float:
    """k_c = π √((m/a)² + (n/b)²) for the (m, n) mode of an a × b rectangle."""
    return math.pi * math.sqrt((m / a) ** 2 + (n / b) ** 2)


def transverse_E(psi: torch.Tensor, hx: float, hy: float, kind: str) -> torch.Tensor:
    """Direction of the transverse electric field, (..., Nx, Ny) → (..., 2, Nx, Ny).

    TM: E_t ∝ ∇E_z = (∂_x ψ, ∂_y ψ).   TE: E_t ∝ ẑ × ∇H_z = (−∂_y ψ, ∂_x ψ).
    """
    dx, dy = torch.gradient(psi, spacing=(hx, hy), dim=(-2, -1))
    return torch.stack([dx, dy], dim=-3) if kind == "TM" else torch.stack([-dy, dx], dim=-3)


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/em/waveguide_modes/tensors.py
"""Build the concrete tensors: the grids for TM (nodes inside the walls) and TE (cell centres)."""

import torch


def grid_size(a: float, b: float, n_b: int) -> tuple[int, int]:
    """(Nx, Ny): n_b points across the short side b, and the same spacing along the long side a."""
    return max(4, round(n_b * a / b)), n_b


def grid(a: float, b: float, nx: int, ny: int, kind: str) -> tuple[torch.Tensor, torch.Tensor, float, float]:
    """Return (x (Nx,), y (Ny,), hx, hy).

    TM (Dirichlet): Nx interior nodes, h = a/(Nx + 1), so the walls x = 0 and x = a are the missing nodes.
    TE (Neumann):   Nx cells of width h = a/Nx, the values sit at the cell centres.
    """
    if kind == "TM":
        hx, hy = a / (nx + 1), b / (ny + 1)
        x = torch.arange(1, nx + 1, dtype=torch.float64) * hx
        y = torch.arange(1, ny + 1, dtype=torch.float64) * hy
    else:
        hx, hy = a / nx, b / ny
        x = (torch.arange(nx, dtype=torch.float64) + 0.5) * hx
        y = (torch.arange(ny, dtype=torch.float64) + 0.5) * hy
    return x, y, hx, hy


def candidate_labels(kind: str, max_index: int = 8) -> list[tuple[int, int]]:
    """The (m, n) pairs a mode of this kind can have: TM needs m, n ≥ 1; TE allows 0 but not both."""
    lowest = 1 if kind == "TM" else 0
    pairs = [(m, n) for m in range(lowest, max_index) for n in range(lowest, max_index)]
    return [p for p in pairs if p != (0, 0)]


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/em/waveguide_modes/solve.py
"""Solve both eigenproblems, merge the TE and TM modes and sort them by cutoff."""

import torch

from . import tensors
from .equation import exact_cutoff, laplacian, modes, transverse_E


def solve(a: float, b: float, n_b: int, n_modes: int) -> list[dict]:
    """Return the n_modes lowest modes as dicts with keys kind, mn, k_c, k_exact, psi (Nx, Ny), E_t (2, Nx, Ny).

    psi is normalised so that Σ_m ψ_m² = 1 (eigh returns orthonormal eigenvectors).
    """
    nx, ny = tensors.grid_size(a, b, n_b)
    found = []
    for kind, boundary in [("TM", "dirichlet"), ("TE", "neumann")]:
        _, _, hx, hy = tensors.grid(a, b, nx, ny, kind)
        L = laplacian(nx, ny, hx, hy, boundary)  # (Nx·Ny, Nx·Ny)
        k_c, psi = modes(L, n_modes, skip_zero=(kind == "TE"))  # (K,), (Nx·Ny, K)
        for k, column in zip(k_c.tolist(), psi.T):
            pattern = column.reshape(nx, ny)  # back from ψ_m to ψ_{ab}
            # name the mode after the exact (m, n) with the nearest cutoff
            mn = min(tensors.candidate_labels(kind), key=lambda p: abs(exact_cutoff(*p, a, b) - k))
            E_t = transverse_E(pattern, hx, hy, kind)
            found.append(
                {"kind": kind, "mn": mn, "k_c": k, "k_exact": exact_cutoff(*mn, a, b), "psi": pattern, "E_t": E_t}
            )
    return sorted(found, key=lambda mode: mode["k_c"])[:n_modes]


def cutoff_frequency_GHz(k_c: float) -> float:
    """f_c = c k_c / 2π with k_c in 1/cm and c = 3 × 10¹⁰ cm/s, in GHz."""
    return 3e10 * k_c / (2 * torch.pi) / 1e9


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/em/waveguide_modes/plot.py
"""Animate the modes one after another: the field pattern oscillating in time, and the list of cutoffs."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from problems._common.plotting import ACCENT, frame_indices, save_gif

FRAMES_PER_MODE = 16


def animate(found: list[dict], frequencies: list[float], a: float, b: float, path: Path) -> Path:
    patterns = [m["psi"].numpy() / np.abs(m["psi"].numpy()).max() for m in found]  # each scaled to ±1
    arrows = [m["E_t"].numpy() for m in found]  # (2, Nx, Ny) each
    nx, ny = patterns[0].shape
    every = max(1, ny // 8)  # one arrow every few cells
    x = (np.arange(nx) + 0.5) * a / nx
    y = (np.arange(ny) + 0.5) * b / ny
    X, Y = np.meshgrid(x[::every], y[::every], indexing="ij")
    n_frames = len(found) * FRAMES_PER_MODE

    fig, (ax, ax_list) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [2.2, 1]})
    image = ax.imshow(
        patterns[0].T, origin="lower", extent=(0, a, 0, b), cmap=plt.get_cmap("RdBu_r", 16), vmin=-1, vmax=1
    )
    quiver = ax.quiver(X, Y, X * 0, Y * 0, color="#111827", pivot="mid", scale=None)
    ax.set_xlabel("x (cm)")
    ax.set_ylabel("y (cm)")
    ax.grid(False)
    title = ax.set_title("")

    # right: every mode with its cutoff; the one on screen is highlighted
    ax_list.axis("off")
    ax_list.set_title("cutoff frequencies", fontsize=10)
    rows = []
    for k, (mode, f) in enumerate(zip(found, frequencies)):
        m, n = mode["mn"]
        text = f"{mode['kind']}$_{{{m}{n}}}$   {f:5.2f} GHz"
        rows.append(ax_list.text(0.05, 0.92 - k * 0.08, text, transform=ax_list.transAxes, fontsize=10))
    ax_list.text(0.05, 0.92 - len(found) * 0.08 - 0.04, "arrows: transverse E", fontsize=8,
                 transform=ax_list.transAxes, color="#6b7280")  # fmt: skip

    def update(frame: int):
        k, phase = divmod(frame, FRAMES_PER_MODE)
        swing = np.cos(2 * np.pi * phase / FRAMES_PER_MODE)  # the mode oscillates as cos(ωt)
        image.set_data(swing * patterns[k].T)
        E = arrows[k][:, ::every, ::every]
        scale = np.abs(E).max() + 1e-12
        quiver.set_UVC(swing * E[0] / scale, swing * E[1] / scale)
        mode = found[k]
        field = "E_z" if mode["kind"] == "TM" else "H_z"
        m, n = mode["mn"]
        title.set_text(f"{mode['kind']}$_{{{m}{n}}}$: colour = {field},  k_c = {mode['k_c']:.3f} /cm")
        for j, row in enumerate(rows):
            row.set_color(ACCENT[1] if j == k else "#374151")
            row.set_fontweight("bold" if j == k else "normal")

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(n_frames, n_frames), path, fps=10)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'a': 2.286,
    'b': 1.016,
    'n_modes': 6,
    'n_grid': 24
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.em.waveguide_modes")]:
    del sys.modules[name]
main = importlib.import_module("problems.em.waveguide_modes.main")
problem_dir = Path("problems/em/waveguide_modes")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
